In [1]:
# 01 — Data Acquisition with Google Earth Engine

#This notebook inspects and exports remotely sensed inputs for satellite-based flood extent and damage-proxy analysis in Nepal.

#**Phase:** 1 — Data acquisition  
#**Status:** Exploratory / event window under verification

In [26]:
from pathlib import Path
import sys

import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError(
        "Could not find the repository root containing the 'src' directory. "
        f"Current working directory: {Path.cwd()}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    AOI_BBOX,
    EVENT_NAME,
    EVENT_DATE,
    EVENT_CAUSE,
    BASELINE_START,
    BASELINE_END,
    POST_EVENT_START,
    POST_EVENT_END,
    S1_POLARIZATIONS,
    S1_ORBIT_PASS,
    S1_RELATIVE_ORBIT,
    S1_BASELINE_DATES,
    S1_POST_EVENT_DATE,
    S2_CLOUD_COVER_MAX,
    EXPORT_SCALE_S1,
    EXPORT_SCALE_S2,
    EXPORT_SCALE_VIIRS,
    EXPORT_SCALE_DEM,
    MAX_PIXELS,
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Python executable: {sys.executable}")
print(f"Event: {EVENT_NAME}")
print(f"Event date: {EVENT_DATE}")

Project root: /Users/nishantanepal/Desktop/Data Analytics/nepal flood sentinel damage
Python executable: /Users/nishantanepal/Desktop/Data Analytics/nepal flood sentinel damage/.venv/bin/python
Event: Rasuwa–Bhote Koshi flash flood
Event date: 2026-08-26


In [28]:
#Initialize Earth Engine
EE_PROJECT_ID = "nepal-flood-sentinel"

try:
    ee.Initialize(project=EE_PROJECT_ID)
    print(
        f"Google Earth Engine initialized successfully: "
        f"{EE_PROJECT_ID}"
    )
except Exception as error:
    print("Earth Engine initialization failed.")
    print(error)

Google Earth Engine initialized successfully: nepal-flood-sentinel


In [29]:
#Defining the AOI
aoi = ee.Geometry.Rectangle(
    [
        AOI_BBOX["min_lon"],
        AOI_BBOX["min_lat"],
        AOI_BBOX["max_lon"],
        AOI_BBOX["max_lat"],
    ]
)

aoi.getInfo()

{'type': 'Polygon',
 'coordinates': [[[85.8, 27.6],
   [86.2, 27.6],
   [86.2, 28.2],
   [85.8, 28.2],
   [85.8, 27.6]]]}

In [30]:
#Displaying the AOI
Map = geemap.Map(
    center=[27.9, 86.0],
    zoom=9,
)

Map.addLayer(
    aoi,
    {"color": "yellow"},
    "Initial AOI",
)

Map

Map(center=[27.9, 86.0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [31]:
#General Sentinel-1 query function
def get_sentinel1_collection(
    start_date,
    end_date,
    orbit_pass=None,
):
    collection = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(
            ee.Filter.eq(
                "instrumentMode",
                "IW",
            )
        )
        .filter(
            ee.Filter.listContains(
                "transmitterReceiverPolarisation",
                "VV",
            )
        )
        .filter(
            ee.Filter.listContains(
                "transmitterReceiverPolarisation",
                "VH",
            )
        )
    )

    if orbit_pass is not None:
        collection = collection.filter(
            ee.Filter.eq(
                "orbitProperties_pass",
                orbit_pass,
            )
        )

    return collection

In [32]:
#Count descending Sentinel-1 scenes
s1_baseline_desc = get_sentinel1_collection(
    BASELINE_START,
    BASELINE_END,
    S1_ORBIT_PASS,
)

s1_post_desc = get_sentinel1_collection(
    POST_EVENT_START,
    POST_EVENT_END,
    S1_ORBIT_PASS,
)

print(
    "Baseline Sentinel-1 scenes, descending:",
    s1_baseline_desc.size().getInfo(),
)

print(
    "Post-event Sentinel-1 scenes, descending:",
    s1_post_desc.size().getInfo(),
)

Baseline Sentinel-1 scenes, descending: 11
Post-event Sentinel-1 scenes, descending: 2


In [33]:
#Count all Sentinel-1 orbits
s1_baseline_all = get_sentinel1_collection(
    BASELINE_START,
    BASELINE_END,
    orbit_pass=None,
)

s1_post_all = get_sentinel1_collection(
    POST_EVENT_START,
    POST_EVENT_END,
    orbit_pass=None,
)

print(
    "Baseline Sentinel-1 scenes, all orbits:",
    s1_baseline_all.size().getInfo(),
)

print(
    "Post-event Sentinel-1 scenes, all orbits:",
    s1_post_all.size().getInfo(),
)

Baseline Sentinel-1 scenes, all orbits: 27
Post-event Sentinel-1 scenes, all orbits: 7


In [34]:
#Sentinel-1 metadata helper
s1_properties = [
    "system:time_start",
    "system:index",
    "orbitProperties_pass",
    "relativeOrbitNumber_start",
    "platform_number",
]

def sentinel1_dataframe(collection):
    features = collection.map(
        lambda image: ee.Feature(
            None,
            image.toDictionary(s1_properties),
        )
    )

    records = features.getInfo()["features"]

    if not records:
        return pd.DataFrame(
            columns=s1_properties + ["acquisition_time"]
        )

    dataframe = pd.DataFrame(
        [
            item["properties"]
            for item in records
        ]
    )

    dataframe["acquisition_time"] = pd.to_datetime(
        dataframe["system:time_start"],
        unit="ms",
        utc=True,
    )

    return dataframe.sort_values(
        "acquisition_time"
    ).reset_index(drop=True)

In [19]:
def get_matched_sentinel1(start_date, end_date):
    return (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(
            ee.Filter.eq(
                "instrumentMode",
                "IW",
            )
        )
        .filter(
            ee.Filter.eq(
                "orbitProperties_pass",
                S1_ORBIT_PASS,
            )
        )
        .filter(
            ee.Filter.eq(
                "relativeOrbitNumber_start",
                S1_RELATIVE_ORBIT,
            )
        )
        .filter(
            ee.Filter.listContains(
                "transmitterReceiverPolarisation",
                "VV",
            )
        )
        .filter(
            ee.Filter.listContains(
                "transmitterReceiverPolarisation",
                "VH",
            )
        )
    )

In [35]:
#Displaying all Sentinel-1 metadata
s1_baseline_df = sentinel1_dataframe(
    s1_baseline_all
)

s1_post_df = sentinel1_dataframe(
    s1_post_all
)

display(s1_baseline_df)
display(s1_post_df)

,orbitProperties_pass,platform_number,relativeOrbitNumber_start,system:time_start,acquisition_time
0,DESCENDING,A,121,1780272675000,2026-06-01 00:11:15+00:00
1,ASCENDING,A,12,1780661631000,2026-06-05 12:13:51+00:00
2,DESCENDING,A,19,1780705162000,2026-06-06 00:19:22+00:00
3,ASCENDING,A,85,1781094114000,2026-06-10 12:21:54+00:00
4,ASCENDING,A,85,1781094139000,2026-06-10 12:22:19+00:00
5,DESCENDING,A,121,1781309475000,2026-06-13 00:11:15+00:00
6,ASCENDING,A,12,1781698430000,2026-06-17 12:13:50+00:00
7,DESCENDING,A,19,1781741962000,2026-06-18 00:19:22+00:00
8,ASCENDING,A,85,1782130913000,2026-06-22 12:21:53+00:00
9,ASCENDING,A,85,1782130938000,2026-06-22 12:22:18+00:00


,orbitProperties_pass,platform_number,relativeOrbitNumber_start,system:time_start,acquisition_time
0,ASCENDING,D,85,1787919676000,2026-08-28 12:21:16+00:00
1,ASCENDING,D,85,1787919701000,2026-08-28 12:21:41+00:00
2,DESCENDING,D,121,1788135037000,2026-08-31 00:10:37+00:00
3,ASCENDING,D,12,1788523993000,2026-09-04 12:13:13+00:00
4,DESCENDING,D,19,1788567524000,2026-09-05 00:18:44+00:00
5,ASCENDING,D,85,1788956476000,2026-09-09 12:21:16+00:00
6,ASCENDING,D,85,1788956501000,2026-09-09 12:21:41+00:00


In [36]:
#Matched Sentinel-1 query
def get_matched_sentinel1(
    start_date,
    end_date,
):
    return (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(
            ee.Filter.eq(
                "instrumentMode",
                "IW",
            )
        )
        .filter(
            ee.Filter.eq(
                "orbitProperties_pass",
                S1_ORBIT_PASS,
            )
        )
        .filter(
            ee.Filter.eq(
                "relativeOrbitNumber_start",
                S1_RELATIVE_ORBIT,
            )
        )
        .filter(
            ee.Filter.listContains(
                "transmitterReceiverPolarisation",
                "VV",
            )
        )
        .filter(
            ee.Filter.listContains(
                "transmitterReceiverPolarisation",
                "VH",
            )
        )
    )

In [37]:
#Create matched Sentinel-1 collections
s1_baseline_matched = get_matched_sentinel1(
    BASELINE_START,
    BASELINE_END,
)

s1_post_matched = get_matched_sentinel1(
    POST_EVENT_START,
    POST_EVENT_END,
)

print(
    "Matched baseline scenes:",
    s1_baseline_matched.size().getInfo(),
)

print(
    "Matched post-event scenes:",
    s1_post_matched.size().getInfo(),
)

Matched baseline scenes: 5
Matched post-event scenes: 1


In [38]:
#Display matched metadata
display(
    sentinel1_dataframe(
        s1_baseline_matched
    )
)

display(
    sentinel1_dataframe(
        s1_post_matched
    )
)

,orbitProperties_pass,platform_number,relativeOrbitNumber_start,system:time_start,acquisition_time
0,DESCENDING,A,19,1780705162000,2026-06-06 00:19:22+00:00
1,DESCENDING,A,19,1781741962000,2026-06-18 00:19:22+00:00
2,DESCENDING,D,19,1782346721000,2026-06-25 00:18:41+00:00
3,DESCENDING,D,19,1783383521000,2026-07-07 00:18:41+00:00
4,DESCENDING,D,19,1784420322000,2026-07-19 00:18:42+00:00


,orbitProperties_pass,platform_number,relativeOrbitNumber_start,system:time_start,acquisition_time
0,DESCENDING,D,19,1788567524000,2026-09-05 00:18:44+00:00


Five scenes met the selected descending-pass/relative-orbit-19 geometry. Four scenes were intentionally used for the initial median baseline; the 25 June Sentinel-1D scene was retained in the availability inventory but excluded from the initial composite. Future sensitivity testing may compare four-scene and five-scene baselines.

In [39]:
#Select the configured baseline dates
def collection_for_dates(
    collection,
    dates,
):
    selected_collections = [
        collection.filterDate(
            ee.Date(date),
            ee.Date(date).advance(
                1,
                "day",
            ),
        )
        for date in dates
    ]

    combined = ee.ImageCollection([])

    for selected_collection in selected_collections:
        combined = combined.merge(
            selected_collection
        )

    return combined


s1_baseline_selected = collection_for_dates(
    s1_baseline_matched,
    S1_BASELINE_DATES,
)

print(
    "Selected baseline scenes:",
    s1_baseline_selected.size().getInfo(),
)

display(
    sentinel1_dataframe(
        s1_baseline_selected
    )
)

Selected baseline scenes: 4


,orbitProperties_pass,platform_number,relativeOrbitNumber_start,system:time_start,acquisition_time
0,DESCENDING,A,19,1780705162000,2026-06-06 00:19:22+00:00
1,DESCENDING,A,19,1781741962000,2026-06-18 00:19:22+00:00
2,DESCENDING,D,19,1783383521000,2026-07-07 00:18:41+00:00
3,DESCENDING,D,19,1784420322000,2026-07-19 00:18:42+00:00


In [40]:
# final selected metadata
print("Final baseline scenes:")
display(
    sentinel1_dataframe(
        s1_baseline_selected
    )
)

print("Final post-event scenes:")
display(
    sentinel1_dataframe(
        s1_post_matched
    )
)

Final baseline scenes:


,orbitProperties_pass,platform_number,relativeOrbitNumber_start,system:time_start,acquisition_time
0,DESCENDING,A,19,1780705162000,2026-06-06 00:19:22+00:00
1,DESCENDING,A,19,1781741962000,2026-06-18 00:19:22+00:00
2,DESCENDING,D,19,1783383521000,2026-07-07 00:18:41+00:00
3,DESCENDING,D,19,1784420322000,2026-07-19 00:18:42+00:00


Final post-event scenes:


,orbitProperties_pass,platform_number,relativeOrbitNumber_start,system:time_start,acquisition_time
0,DESCENDING,D,19,1788567524000,2026-09-05 00:18:44+00:00


In [41]:
#SAR composites and change
baseline_s1 = (
    s1_baseline_selected
    .select(
        ["VV", "VH"]
    )
    .median()
    .clip(aoi)
)

post_s1 = (
    s1_post_matched
    .select(
        ["VV", "VH"]
    )
    .median()
    .clip(aoi)
)

vv_change = (
    post_s1
    .select("VV")
    .subtract(
        baseline_s1.select("VV")
    )
    .rename("VV_change")
)

print(
    "Baseline bands:",
    baseline_s1.bandNames().getInfo(),
)

print(
    "Post-event bands:",
    post_s1.bandNames().getInfo(),
)

print(
    "Change band:",
    vv_change.bandNames().getInfo(),
)

Baseline bands: ['VV', 'VH']
Post-event bands: ['VV', 'VH']
Change band: ['VV_change']


In [42]:
#SAR layers
sar_vis = {
    "min": -25,
    "max": 0,
}

change_vis = {
    "min": -8,
    "max": 8,
    "palette": [
        "b2182b",
        "f7f7f7",
        "2166ac",
    ],
}

Map = geemap.Map(
    center=[27.9, 86.0],
    zoom=10,
)

Map.addLayer(
    baseline_s1.select("VV"),
    sar_vis,
    "Baseline Sentinel-1 VV — median",
)

Map.addLayer(
    post_s1.select("VV"),
    sar_vis,
    "Post-event Sentinel-1 VV — 2026-09-05",
)

Map.addLayer(
    vv_change,
    change_vis,
    "VV backscatter change — post minus baseline",
)

Map.addLayer(
    aoi,
    {"color": "yellow"},
    "Initial AOI",
)

Map

Map(center=[27.9, 86.0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [50]:
phase1_map = geemap.Map(
    center=[27.9, 86.0],
    zoom=10,
)

phase1_map.addLayer(
    vv_change,
    change_vis,
    "VV change — 2026-09-05 minus baseline",
)

phase1_map.addLayer(
    aoi,
    {"color": "yellow"},
    "Initial AOI",
)

phase1_map

Map(center=[27.9, 86.0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [52]:
change_vis_final = {
    "min": -1.5,
    "max": 1.5,
    "palette": [
        "8b0000",
        "ff0000",
        "ffffff",
        "0000ff",
        "00008b",
    ],
}

phase1_map = geemap.Map(
    center=[27.9, 86.0],
    zoom=9,
)

phase1_map.addLayer(
    vv_change,
    change_vis_final,
    "VV change — 2026-09-05 minus baseline",
)

phase1_map.addLayer(
    aoi,
    {
        "color": "yellow",
        "fillColor": "00000000",
    },
    "Initial AOI",
)

phase1_map

Map(center=[27.9, 86.0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

### Figure: Initial study-area context
The yellow rectangle shows the provisional Bhote Koshi/Sindhupalchok investigation AOI used for Phase 1 satellite-scene availability checks. The background provides geographic and terrain context. The AOI will be narrowed during Phase 2 before flood classification and damage-proxy estimation.

In [53]:
phase1_map.to_html(
    str(
        PROJECT_ROOT
        / "figures"
        / "phase1_sentinel1_vv_change.html"
    )
)

print("Saved final Phase 1 map.")

Saved final Phase 1 map.


In [43]:
#Sentinel-2 query
def get_sentinel2_collection(
    start_date,
    end_date,
):
    return (
        ee.ImageCollection(
            "COPERNICUS/S2_SR_HARMONIZED"
        )
        .filterBounds(aoi)
        .filterDate(
            start_date,
            end_date,
        )
        .filter(
            ee.Filter.lte(
                "CLOUDY_PIXEL_PERCENTAGE",
                S2_CLOUD_COVER_MAX,
            )
        )
    )


s2_baseline = get_sentinel2_collection(
    BASELINE_START,
    BASELINE_END,
)

s2_post = get_sentinel2_collection(
    POST_EVENT_START,
    POST_EVENT_END,
)

print(
    "Baseline Sentinel-2 scenes:",
    s2_baseline.size().getInfo(),
)

print(
    "Post-event Sentinel-2 scenes:",
    s2_post.size().getInfo(),
)

Baseline Sentinel-2 scenes: 5
Post-event Sentinel-2 scenes: 0


In [44]:
#Sentinel-2 unfiltered post-event check
s2_post_all_clouds = (
    ee.ImageCollection(
        "COPERNICUS/S2_SR_HARMONIZED"
    )
    .filterBounds(aoi)
    .filterDate(
        POST_EVENT_START,
        POST_EVENT_END,
    )
)

print(
    "Post-event Sentinel-2 scenes, all cloud levels:",
    s2_post_all_clouds.size().getInfo(),
)

Post-event Sentinel-2 scenes, all cloud levels: 22


In [45]:
#VIIRS availability
viirs = (
    ee.ImageCollection(
        "NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG"
    )
    .filterBounds(aoi)
)

viirs_count = viirs.size().getInfo()

print(
    f"VIIRS monthly images available for AOI: "
    f"{viirs_count}"
)

VIIRS monthly images available for AOI: 152


In [46]:
#DEM and slope
dem = (
    ee.Image(
        "USGS/SRTMGL1_003"
    )
    .select("elevation")
    .clip(aoi)
)

slope = ee.Terrain.slope(dem)

terrain_map = geemap.Map(
    center=[27.9, 86.0],
    zoom=10,
)

terrain_map.addLayer(
    dem,
    {
        "min": 500,
        "max": 7000,
        "palette": [
            "0b3d02",
            "8cc751",
            "e6d27a",
            "b75d2a",
            "ffffff",
        ],
    },
    "SRTM elevation",
)

terrain_map.addLayer(
    slope,
    {
        "min": 0,
        "max": 60,
        "palette": [
            "1a9850",
            "fee08b",
            "d73027",
        ],
    },
    "Slope (degrees)",
)

terrain_map.addLayer(
    aoi,
    {"color": "yellow"},
    "Initial AOI",
)

terrain_map

Map(center=[27.9, 86.0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [47]:
#Export interactive Phase 1 map
FIGURES_DIR = PROJECT_ROOT / "figures"
FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

phase1_html_path = (
    FIGURES_DIR
    / "phase1_sentinel1_vv_change.html"
)

phase1_map = geemap.Map(
    center=[27.9, 86.0],
    zoom=10,
)

phase1_map.addLayer(
    vv_change,
    change_vis,
    "VV change",
)

phase1_map.addLayer(
    aoi,
    {"color": "yellow"},
    "AOI",
)

phase1_map.to_html(
    str(phase1_html_path)
)

print(
    f"Saved interactive Phase 1 map to: "
    f"{phase1_html_path}"
)

Saved interactive Phase 1 map to: /Users/nishantanepal/Desktop/Data Analytics/nepal flood sentinel damage/figures/phase1_sentinel1_vv_change.html


In [49]:
phase1_map = geemap.Map(
    center=[27.9, 86.0],
    zoom=10,
)

phase1_map.addLayer(
    vv_change,
    change_vis,
    "VV change — 2026-09-05 minus baseline",
)

phase1_map.addLayer(
    aoi,
    {"color": "yellow"},
    "Initial AOI",
)

phase1_map

Map(center=[27.9, 86.0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [48]:
print("Phase 1 acquisition summary")
print("---------------------------")
print(f"Event: {EVENT_NAME}")
print(f"Event date: {EVENT_DATE}")
print(f"AOI: {AOI_BBOX}")
print(
    "Baseline Sentinel-1 scenes, all orbits:",
    s1_baseline_all.size().getInfo(),
)
print(
    "Post-event Sentinel-1 scenes, all orbits:",
    s1_post_all.size().getInfo(),
)
print(
    "Selected baseline scenes:",
    s1_baseline_selected.size().getInfo(),
)
print(
    "Selected post-event scenes:",
    s1_post_matched.size().getInfo(),
)
print(
    "Baseline Sentinel-2 scenes:",
    s2_baseline.size().getInfo(),
)
print(
    "Post-event Sentinel-2 scenes under cloud threshold:",
    s2_post.size().getInfo(),
)
print(
    "VIIRS monthly images:",
    viirs_count,
)
print(
    "Primary Sentinel-1 geometry:",
    f"{S1_ORBIT_PASS}, relative orbit "
    f"{S1_RELATIVE_ORBIT}",
)
print(
    "Post-event Sentinel-1 date:",
    S1_POST_EVENT_DATE,
)

Phase 1 acquisition summary
---------------------------
Event: Rasuwa–Bhote Koshi flash flood
Event date: 2026-08-26
AOI: {'min_lon': 85.8, 'min_lat': 27.6, 'max_lon': 86.2, 'max_lat': 28.2}
Baseline Sentinel-1 scenes, all orbits: 27
Post-event Sentinel-1 scenes, all orbits: 7
Selected baseline scenes: 4
Selected post-event scenes: 1
Baseline Sentinel-2 scenes: 5
Post-event Sentinel-2 scenes under cloud threshold: 0
VIIRS monthly images: 152
Primary Sentinel-1 geometry: DESCENDING, relative orbit 19
Post-event Sentinel-1 date: 2026-09-05
